# 2.4 — Réentraînement à 1024×1024

Même configuration que 2.2, avec `imgsz=1024` et `batch=8` (contrainte mémoire P100 16 Go).
**Les poids sont déjà entraînés** (`runs/yolov8m_epi_1024-2/weights/best.pt`, epoch 79) — relancer ce notebook uniquement pour reproduire l'entraînement.
*(Note : un premier essai `yolov8m_epi_1024` interrompu après 2 epochs a laissé Ultralytics nommer le run complet `-2`.)*
Motivation : voir doc 03 (section « Révision du choix ») et doc 04 (section 2.4).

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import torch

DATASET_ROOT = Path('/root/Projet_Image/SH17dataset')
RUNS_DIR     = Path('/root/Projet_Image/runs')

print(f'CUDA disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
assert (DATASET_ROOT / 'sh17.yaml').exists(), 'sh17.yaml introuvable'

In [ ]:
model = YOLO('yolov8m.pt')

results = model.train(
    data            = str(DATASET_ROOT / 'sh17.yaml'),
    epochs          = 100,
    patience        = 20,
    imgsz           = 1024,   # résolution augmentée vs 640 en 2.2
    batch           = 8,      # réduit vs 16 en 2.2 — contrainte mémoire à 1024×1024
    device          = 0,
    freeze          = 10,
    optimizer       = 'SGD',
    lr0             = 0.01,
    lrf             = 0.01,
    momentum        = 0.937,
    weight_decay    = 0.0005,
    cos_lr          = True,
    warmup_epochs   = 3,
    warmup_momentum = 0.8,
    label_smoothing = 0.1,
    hsv_h           = 0.015,
    hsv_s           = 0.7,
    hsv_v           = 0.4,
    fliplr          = 0.5,
    flipud          = 0.0,
    degrees         = 12.0,
    translate       = 0.1,
    scale           = 0.5,
    perspective     = 0.0005,
    mosaic          = 1.0,
    erasing         = 0.4,
    project         = str(RUNS_DIR),
    name            = 'yolov8m_epi_1024',
    save            = True,
    plots           = True,
)

In [ ]:
import pandas as pd
from IPython.display import Image as IPImage, display

run_dir = RUNS_DIR / 'yolov8m_epi_1024-2'

csv_path = run_dir / 'results.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    best = df.loc[df['metrics/mAP50(B)'].idxmax()]
    print(f"Meilleure epoch : {int(best['epoch'])}")
    print(f"mAP50 val      : {best['metrics/mAP50(B)']:.3f}")
    print(f"mAP50-95 val   : {best['metrics/mAP50-95(B)']:.3f}")
    print(f"Precision val  : {best['metrics/precision(B)']:.3f}")
    print(f"Recall val     : {best['metrics/recall(B)']:.3f}")

for plot in ['results.png', 'confusion_matrix_normalized.png']:
    p = run_dir / plot
    if p.exists():
        display(IPImage(str(p)))

## Évaluation sur le split test (imgsz=1024)

Cette section évalue le modèle retenu (`yolov8m_epi_1024-2/weights/best.pt`, epoch 79) sur le **split test**, à `imgsz=1024` (cohérent avec son entraînement). Elle reproduit le tableau de `documentation/04.Evaluation.md` section 2.4 (mAP50 = 0.697, mAP50-95 = 0.412, Précision = 0.798, Rappel = 0.616, F1 = 0.695).

*(Le notebook `2_3_evaluation.ipynb` évalue séparément le modèle 640×640 (`yolov8m_epi/weights/best.pt`) à `imgsz=640`, et reproduit le tableau de la section 2.3.)*

In [ ]:
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

model_test = YOLO(str(RUNS_DIR / 'yolov8m_epi_1024-2' / 'weights' / 'best.pt'))

results_test = model_test.val(
    data    = str(DATASET_ROOT / 'sh17.yaml'),
    split   = 'test',
    imgsz   = 1024,
    batch   = 8,
    device  = DEVICE,
    verbose = False,
    plots   = False,
)

In [ ]:
map50     = float(results_test.box.map50)
map50_95  = float(results_test.box.map)
precision = float(results_test.box.mp)
recall    = float(results_test.box.mr)
f1_global = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"mAP50      : {map50:.3f}")
print(f"mAP50-95   : {map50_95:.3f}")
print(f"Précision  : {precision:.3f}")
print(f"Rappel     : {recall:.3f}")
print(f"F1 (global): {f1_global:.3f}")

In [ ]:
names       = model_test.names
class_idx   = results_test.box.ap_class_index
precision_c = results_test.box.p
recall_c    = results_test.box.r
f1_c        = results_test.box.f1
ap50_c      = results_test.box.ap50
ap_c        = results_test.box.ap

rows = [
    {
        'Classe':    names[i],
        'Précision': round(float(p), 3),
        'Rappel':    round(float(r), 3),
        'F1':        round(float(f), 3),
        'mAP50':     round(float(a50), 3),
        'mAP50-95':  round(float(a), 3),
    }
    for i, p, r, f, a50, a in zip(class_idx, precision_c, recall_c, f1_c, ap50_c, ap_c)
]
rows.append({
    'Classe':    'GLOBAL (moyenne)',
    'Précision': round(precision, 3),
    'Rappel':    round(recall, 3),
    'F1':        round(f1_global, 3),
    'mAP50':     round(map50, 3),
    'mAP50-95':  round(map50_95, 3),
})

table = pd.DataFrame(rows)
table